# Task 6: License Dataset Analysis

Action: Load and explore the Ontario elevator license dataset to prepare it for use in future modules.

## Setup: Import libraries and load license.csv
Use pandas to load data/license.csv into a DataFrame; display shape and head for context.

In [1]:
import pandas as pd

license_df = pd.read_csv("../data/license.csv")

license_df.shape

(45383, 11)

In [2]:
license_df.head(5)

,ElevatingDevicesNumber,LocationoftheElevatingDevice,ElevatingDevicesLicenseNumber,LICENSESTATUS,LICENSEEXPIRYDATE,LICENSEHOLDER,LICENSEHOLDERACCOUNTNUMBER,LICENSEHOLDERADDRESS,BILLINGCUSTOMER,BILLINGADDRESS,BILLINGACCOUNT
0,10,111 WELLESLEY ST W TORONTO M7A 1A2 ON CA,EDLIC-000010,ACTIVE,28-Apr-17,LEGISLATIVE ASSEMBLY OF ONTARIO ATTN: JOHN ED...,data redacted,99 WELLESLEY ST W WHITNEY BLOCK ROOM 2540 TOR...,LEGISLATIVE ASSEMBLY OF ONTARIO ATTN: JOHN ED...,99 WELLESLEY ST W WHITNEY BLOCK ROOM 2540 TOR...,data redacted
1,100,1804 HIGHWAY 2 E BROCKVILLE K6V 5T1 ON CA,170719,BY REQUEST,12-Dec-14,INFRASTRUCTURE ONTARIO AND LANDS CORPORATION,data redacted,18 KING ST E TORONTO ON M5C 1C4 CA,CB RICHARD ELLIS GLOBAL CORPORATE SERVICES,333 PRESTON ST 7TH FLR PRESTON SQUARE TOWER 1 ...,data redacted
2,10047,162 PEMBROKE ST W PEMBROKE K8A 5M8 ON CA,EDLIC-010047,BY REQUEST,15-Mar-08,PROFAC MANAGEMENT GROUP LTD,data redacted,304 THE EAST MALL P.O. # 653058-Y3-20610 TORON...,PROFAC MANAGEMENT GROUP LTD,304 THE EAST MALL P.O. # 653058-Y3-20610 TORON...,data redacted
3,10054,541 SUSSEX DR OTTAWA K1N 6Z6 ON CA,EDLIC-010054,BY REQUEST,01-Oct-05,DEPARTMENT OF PUBLIC WORKS & GOVERNMENT SERVIC...,data redacted,4900 YONGE ST 11TH FLOOR TORONTO ON M2N 6A6 CA,DEPARTMENT OF PUBLIC WORKS & GOVERNMENT SERVIC...,"4900 YONGE ST 11TH FLOOR TORONTO, ON, M2N 6A6, CA",data redacted
4,1009,404 MAIN ST WOODSTOCK N4S 7X5 ON CA,EDLIC-001009,ACTIVE,15-Jul-17,AGRIBRANDS PURINA CANADA INC,data redacted,404 MAIN ST PO BOX 250 WOODSTOCK ON N4S 7X5 CA,AGRIBRANDS PURINA CANADA INC,"404 MAIN ST PO BOX 250 WOODSTOCK, ON, N4S 7X5, CA",data redacted


## (a) Identify unique elevator identifier with uniqueness checks
Inspect candidate ID-like columns, compute distinct counts and uniqueness; select the best identifier and justify with code outputs.

In [3]:
pd.DataFrame({"column": license_df.columns, "dtype": license_df.dtypes.astype(str)})

,column,dtype
ElevatingDevicesNumber,ElevatingDevicesNumber,int64
LocationoftheElevatingDevice,LocationoftheElevatingDevice,str
ElevatingDevicesLicenseNumber,ElevatingDevicesLicenseNumber,str
LICENSESTATUS,LICENSESTATUS,str
LICENSEEXPIRYDATE,LICENSEEXPIRYDATE,str
LICENSEHOLDER,LICENSEHOLDER,str
LICENSEHOLDERACCOUNTNUMBER,LICENSEHOLDERACCOUNTNUMBER,str
LICENSEHOLDERADDRESS,LICENSEHOLDERADDRESS,str
BILLINGCUSTOMER,BILLINGCUSTOMER,str
BILLINGADDRESS,BILLINGADDRESS,str


In [4]:
candidate_cols = [
    col
    for col in license_df.columns
    if any(token in col.lower() for token in ["id", "number", "license", "licence", "device", "elevator"])
]

uniqueness = []
for col in candidate_cols:
    series = license_df[col]
    uniqueness.append(
        {
            "column": col,
            "non_null_count": int(series.notna().sum()),
            "unique_count": int(series.nunique(dropna=True)),
            "is_unique": bool(series.is_unique),
        }
    )

pd.DataFrame(uniqueness).sort_values("unique_count", ascending=False)

,column,non_null_count,unique_count,is_unique
0,ElevatingDevicesNumber,45383,45383,True
2,ElevatingDevicesLicenseNumber,45383,45383,True
1,LocationoftheElevatingDevice,45340,23442,False
5,LICENSEHOLDER,45383,16363,False
7,LICENSEHOLDERADDRESS,45383,13276,False
4,LICENSEEXPIRYDATE,45383,1118,False
3,LICENSESTATUS,45383,11,False
6,LICENSEHOLDERACCOUNTNUMBER,45383,1,False


**Justification:**
Based on the uniqueness checks above, the identifier to track each elevator is the column with `is_unique = True` and `unique_count` equal to the dataset row count (typically `ElevatingDevicesNumber`). This means every elevator has a distinct identifier with no duplicates or missing values, making it suitable for merging across Module 2 datasets.

## (b) Extract country and province from location column
The dataset contains a location column that combines geographic information. Extract the country and state/province into two new columns using a pandas string method and identify where the majority of elevators are located.

In [5]:
license_df[["province", "country"]] = license_df["LocationoftheElevatingDevice"].str.extract(
    r"\s([A-Z]{2})\s+([A-Z]{2})\s*$"
)

print(license_df["country"].value_counts(dropna=False))
print()
print(license_df["province"].value_counts(dropna=False))
print()

majority_country = license_df["country"].value_counts().idxmax()
majority_province = license_df["province"].value_counts().idxmax()
f"Majority location: {majority_province}, {majority_country}"

NameError: name 'license_df' is not defined